# ChessMind – trenowanie modelu v1

---

## 1. Cel i zakres

- Zbudowanie lekkiej funkcji oceny pozycji (eval) skorelowanej z ocenami Stockfisha, do pracy z płytkim przeszukiwaniem (głębokość 1–3).
- Przejrzysty przebieg prac: shardy CSV → trening → zapis najlepszego modelu → test gry.
- Wersja v1 świadomie ogranicza złożoność cech i architektury, aby uruchomienia były szybkie, powtarzalne i łatwe do diagnozowania.

---

## 2. Dane i podział

- Dane wejściowe: shardy CSV zawierające FEN oraz docelowe oceny w centipionach (cp) obliczone przez Stockfisha.
- Proporcje faz gry: 30% debiut (OPEN), 30% środek (MID), 40% końcówka (END) – pozwala utrzymać widoczność końcówek w trakcie uczenia.
- Walidacja wydzielona na poziomie shardów; brak mieszania pozycji między zbiorem treningowym i walidacyjnym.
- Charakter próbki: dominują pozycje „typowe” (zbalansowany materiał, brak natychmiastowych kombinacji), rzadziej występują pozycje ekstremalne (mat w 1, duża przewaga/strata materiału).

---

## 3. Reprezentacja pozycji (v1)

- Cechy ogólne: kto jest na ruchu (side-to-move), wskaźnik fazy gry, proste cechy materiałowe (suma figur stron oraz różnica materiału).
- Cechy techniczne występujące w części przebiegów: głębokość/czas evaluacji z generatora danych; są to informacje „meta”, niezwiązane bezpośrednio z geometrią pozycji.
- Braki zamierzone w v1: brak pełnej geometrii planszy 8×8 (położenia figur), brak explicite opisanej struktury pionowej i bezpieczeństwa króla. Dzięki temu v1 jest szybkie, lecz jego obraz pozycji pozostaje uproszczony.

---

## 4. Model i przebieg treningu

- Architektura: wielowarstwowy perceptron (MLP) o niewielkiej głębokości; pojedyncza skalarna predykcja (eval).
- Optymalizacja: typowy przebieg z optymalizatorem AdamW; liczba epok rzędu kilkunastu–kilkudziesięciu; zapisywany „best checkpoint” według metryki walidacyjnej.
- Metryki obserwowane w logach: stabilny spadek strat na początku, spłaszczanie w dalszej części; czas jednej epoki utrzymuje się w wąskim przedziale, co świadczy o równomiernym strumieniu danych.

---

## 5. Walidacja i interpretacja

- Monitorowane wartości: train_loss, val_loss, val_MAE_cp, czas epoki oraz liczba pozycji walidacyjnych.
- Typowy wynik walidacji: MAE około 120 cp; dobra korelacja w środkowym zakresie skali, większy rozrzut na krańcach (bardzo wysokie i bardzo niskie oceny).
- Wykres „Pred vs Target (val)”: gęsty pas punktów wzdłuż przekątnej w centrum zakresu, rozstrzelenie rośnie przy ekstremalnych wartościach – zgodne z oczekiwaniami dla uproszczonej reprezentacji.

---

## 6. Wykorzystanie oceny w grze

- Algorytm gry: płytki negamax (1–3 półruchy), który używa predykcji modelu jako funkcji oceny w liściach drzewa.
- Kolejność ruchów: oparta o standardową enumerację legalnych posunięć z biblioteki szachowej; brak dodatkowych preferencji wynikających z geometrii.
- Zachowanie charakterystyczne dla v1: gdy kilka ruchów otrzymuje zbliżone oceny, wybór bywa wrażliwy na kolejność iteracji i drobne różnice w predykcjach, co może prowadzić do powtarzalnych schematów ruchów.

---

## 7. Obserwacje z gier testowych (na podstawie logów)

- partia.txt (tryb ciągły, czarne): częste przejścia skoczka na krawędź (np. Nh6) i powrót (Ng8), a także wielokrotne przestawianie wieży między g8 i h8. Brak wyraźnego postępu rozwojowego; ruchy oscylacyjne w strefie króla.
- partia_z_czlowiekiem.txt (gra na żywo): utrwalony motyw „tam–z powrotem” wieżą (Rg8 ↔ Rh8) przy aktywnych działaniach białych. W zapisie pojawiały się też komunikaty o niepoprawnym formacie ruchów po stronie przeciwnika – dotyczy interfejsu testowego, nie samego modelu.
- partia_z_czlowiekiem_ale_inny_case.txt (pozycja ustawiona w środku gry): szybkie i poprawne rozpoznanie gotowego motywu matowego „Qxh2#”. W oczywistych sytuacjach taktycznych płytka głębokość bywa wystarczająca.
- partia_z_czlowiekiem_v2.txt (dłuższa partia): początek naturalny, natomiast w środku gry mniejsza wrażliwość na bezpieczeństwo króla i koordynację figur; w końcówce nie zawsze odczytywane są proste sekwencje obronne przeciw groźbom mata.
- partia_koncowka.txt (K+Q vs K, białe): po serii szachów następuje „Qg6+?”, po czym „Kxg6” i remis przez niedostatek materiału. Zapis ilustruje, że w wymuszonych końcówkach kontrola wymian nie zawsze jest wystarczająca mimo przewagi.
- partia_koncowka_mat_w_1.txt (mata w 1 w układzie wieżowym): zamiast natychmiastowego zakończenia partii następuje cykliczne przestawianie wież; rośnie liczba półruchów bez materialnego postępu na planszy.

---

## 8. Ograniczenia znane w v1

- Ograniczona informacja przestrzenna: bez bezpośredniego kodowania położenia figur na 64 polach trudniej różnicować ruchy, które materiałowo wydają się podobne.
- Płaskie rozstrzygnięcia: przy zbliżonych ocenach kilku ruchów wybór jest silnie zależny od kolejności generowania ruchów oraz minimalnych różnic w predykcjach.
- Zachowanie w końcówkach: w sekwencjach wymuszonych nie zawsze zachowana jest kontrola nad wymianami, co może redukować wcześniej wypracowaną przewagę.
- Interpretacja wykresów: większy rozrzut na krańcach skali cp przekłada się na mniejszą precyzję oceny skrajnych pozycji.

---

## 9. Słowniczek i notatki interpretacyjne

- Centipion (cp): jednostka oceny pozycji; 100 cp ≈ wartość piona.
- MAE [cp]: średni błąd bezwzględny między predykcją modelu a celem; łatwy do interpretacji w praktyce.
- FEN: tekstowa reprezentacja pozycji szachowej używana w shardach i interfejsach testowych.
- Logi treningu: zawierają m.in. liczbę shardów, urządzenie (cpu/cuda), metryki po epokach, informację o zapisie najlepszego modelu.
- Logi gry: zawierają planszę tekstową, FEN, ruchy w SAN/uci oraz ocenę czasu odpowiedzi; wskazują także komunikaty o niepoprawnych wejściach po stronie interfejsu.

---

## 10. Podsumowanie

- v1 pełni rolę lekkiej funkcji oceny do szybkich eksperymentów z płytkim przeszukiwaniem.
- W testach łączy umiejętność wykrywania prostych motywów taktycznych z przewidywalnymi ograniczeniami wynikającymi z oszczędnej reprezentacji pozycji.
- Zaobserwowane zjawiska w partiach (oscylacje figur, nieuchwycenie mata w 1, oddanie hetmana w prostej końcówce) są spójne z zakresem i założeniami wersji v1.

---

## 11. Wykresy

![Loss Curve](../../plots/model_v1/loss_curve.png)
![Predicted vs Target](../../plots/model_v1/pred_vs_target.png)
![Residual Histogram](../../plots/model_v1/residual_hist.png)

---